### Group access management

- **Data Analysts**: Read-only access to "Gold" data layers, Genie Spaces, and permissions to use SQL warehouses.
- **Data Engineers**: Permissions to create and manage clusters, Lakeflow pipelines, and write access to "Bronze" and "Silver" data layers.
- **Machine Learning Engineers**: Entitlements for MLflow experiment tracking, Model Serving, and GPU-enabled cluster policies.
- **DevOps**: Access to configure service principals, CI/CD pipeline management, and audit log monitoring.
- **Admins**: Full control over workspace settings, identity management, and Unity Catalog metastores.

#### SELECT permission to Data Analysts

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import iam, catalog

w = WorkspaceClient()

# 1. Create a New User
new_user = w.users.create(
    user_name="analyst_user@example.com",
    display_name="Analyst User"
)
print(f"Created user: {new_user.display_name} (ID: {new_user.id})")

In [0]:
# 2. Create a New Group
analyst_group = w.groups.create(
    display_name="Data_Analysts_Team"
)
print(f"Created group: {analyst_group.display_name} (ID: {analyst_group.id})")

In [0]:
# 3. Add User to the Group
# Note: Use Patch to update members without overwriting the entire group
w.groups.patch(
    id=analyst_group.id,
    operations=[
        iam.Patch(
            op=iam.PatchOp.ADD,
            path="members",
            value=[{"value": new_user.id}]
        )
    ]
)
print(f"Added {new_user.user_name} to {analyst_group.display_name}")

In [0]:
# 4. Assign Permissions in Unity Catalog
# Granting USE_CATALOG and USE_SCHEMA is required for data access
catalog_name = "end_to_end_retail"
schema_name = "db_gold"
table_name = "tbg_customer_address_clustering"

w.grants.update(
    full_name=catalog_name,
    securable_type=catalog.SecurableType.CATALOG,
    changes=[
        catalog.PermissionsChange(
            principal=analyst_group.display_name,
            add=[catalog.Privilege.USE_CATALOG]
        )
    ]
)

In [0]:
# Grant SELECT access to a specific table
w.grants.update(
    full_name=f"{catalog_name}.{schema_name}.{table_name}",
    securable_type=catalog.SecurableType.TABLE,
    changes=[
        catalog.PermissionsChange(
            principal=analyst_group.display_name,
            add=[catalog.Privilege.SELECT]
        )
    ]
)
print(f"Granted SELECT on {table_name} to {analyst_group.display_name}")

##### Function

In [0]:
# Managing permissions for a specific group
def grant_read_access(catalog_name, principal):
    spark.sql(f"GRANT USAGE ON CATALOG {catalog_name} TO `{principal}`")
    spark.sql(f"GRANT USAGE ON SCHEMA {catalog_name}.default TO `{principal}`")
    spark.sql(f"GRANT SELECT ON SCHEMA {catalog_name}.default TO `{principal}`")
    print(f"Read access granted to {principal}")

grant_read_access("gold_layer", "data_analysts_group")


##### Mask function

- end_to_end_retail.db_bronze.tbb_bright_home_orders_enriched

In [0]:
%sql
SELECT email FROM end_to_end_retail.db_bronze.tbb_bright_home_orders_enriched limit 1

In [0]:
%sql
-- 1. Create a masking function (SQL)
CREATE OR REPLACE FUNCTION email_mask(email STRING)
  RETURN IF(is_account_group_member('admin'), email, '****@****.com');

-- 2. Apply it to a table (SQL)
ALTER TABLE end_to_end_retail.db_bronze.tbb_bright_home_orders_enriched 
ALTER COLUMN email SET MASK email_mask;

In [0]:
%sql
SELECT email_mask FROM end_to_end_retail.db_bronze.tbb_bright_home_orders_enriched limit 1

##### Table extended functions

In [0]:
# Discovering metadata (Table Properties and Comments)
table_info = spark.sql("DESCRIBE TABLE EXTENDED end_to_end_retail.db_bronze.tbb_bright_home_orders_enriched")

# Displaying table comments to ensure data discovery documentation exists
display(table_info.filter(table_info.col_name == "Comment"))

# Checking Table Constraints (Important for governance)
constraints = spark.sql("SHOW TBLPROPERTIES end_to_end_retail.db_bronze.tbb_bright_home_orders_enriched")
display(constraints)


#### 1. Python Script: Tagging PII, PHI, owner

In [0]:
# This helps data stewards find PII quickly
def tag_tables_with_pii(table_list):
    for table in table_list:
        spark.sql(f"ALTER TABLE {table} SET TAGS ('contains_pii' = 'true', 'data_sensitivity' = 'high')")

# Example usage:
# tag_tables_with_pii(["main.sensitive_data.raw_users", "main.sensitive_data.customer_info"])

##### Script: Applying Multi-Level Tags

In [0]:
# 1. Tagging a Schema (Broad Governance)
spark.sql("""
  ALTER SCHEMA main.healthcare_data 
  SET TAGS ('data_classification' = 'PHI', 'owner' = 'clinical_ops')
""")

# 2. Tagging a Table (Operational Metadata)
spark.sql("""
  ALTER TABLE main.healthcare_data.patient_logs 
  SET TAGS (
    'refresh_frequency' = 'Daily',
    'contains_pii' = 'true',
    'retention_period' = '7_years'
  )
""")

# 3. Tagging a Column (Fine-Grained Discovery)
# Note: Column tags are excellent for identifying specific PII types
spark.sql("""
  ALTER TABLE main.healthcare_data.patient_logs 
  ALTER COLUMN patient_ssn SET TAGS ('pii_type' = 'ssn', 'masking_applied' = 'true')
""")

#### 2. Defining Secure Masking Functions

In [0]:
%sql
-- Create a schema for governance logic
CREATE SCHEMA IF NOT EXISTS end_to_end_retail.governance_functions;

In [0]:
%sql
-- Email Masking: Only 'hr_admins' see the full email; others see the domain only.
CREATE OR REPLACE FUNCTION end_to_end_retail.governance_functions.email_mask(email STRING)
RETURN IF(is_account_group_member('hr_admins'), email, regexp_replace(email, '^.*$', '****@****.***'));

-- Credit Card Masking: Show only the last 4 digits
CREATE OR REPLACE FUNCTION end_to_end_retail.governance_functions.cc_mask(cc_num STRING)
RETURN IF(is_account_group_member('finance_admins'), cc_num, concat('****-****-****-', right(cc_num, 4)));

-- PHI Redaction: Completely hide SSN/Health IDs for non-medical staff
CREATE OR REPLACE FUNCTION end_to_end_retail.governance_functions.phi_redact(ssn STRING)
RETURN IF(is_account_group_member('medical_staff'), ssn, 'REDACTED');


#### 3. Row-Level Security (Data Filtering)

- https://docs.databricks.com/aws/en/data-governance/unity-catalog/abac/policies 

- Due to the complexity of data architecture, any "rule" should be at the policy level. 

In [0]:
%sql
-- Create a function to filter rows by region
CREATE OR REPLACE FUNCTION governance_functions.region_filter(region STRING)
RETURN is_account_group_member(concat('group_', region)) OR is_account_group_member('admin');

-- Apply the filter to the table
ALTER TABLE main.sensitive_data.patient_records 
SET ROW FILTER governance_functions.region_filter ON (region);


#### 4. Applying Masks to Tables

In [0]:
%sql
-- Apply masking to a customer PII table
ALTER TABLE end_to_end_retail.db_bronze.tbb_bright_home_orders_enriched 
ALTER COLUMN email SET MASK end_to_end_retail.governance_functions.email_mask;

-- ALTER TABLE main.sensitive_data.customers 
-- ALTER COLUMN credit_card SET MASK governance_functions.cc_mask;

-- ALTER TABLE main.sensitive_data.patient_records 
-- ALTER COLUMN ssn SET MASK governance_functions.phi_redact;


In [0]:
%sql
SELECT email FROM end_to_end_retail.db_bronze.tbb_bright_home_orders_enriched limit 1

#### 5. Summary of Best Practices

- Feature	Implementation	Governance Goal
- Column Masking	SET MASK	Protects PII while allowing aggregate analysis.
- Row Filtering	SET ROW FILTER	Restricts data access based on geography or department.
- Tags	SET TAGS	Enables "Data Discovery" and compliance auditing.
- Volumes	GRANT READ	Secures raw unstructured PHI files (scans/images).

### Security by Design approach. 

Tagging:

- Role-Based Access (RBAC) 
- Redaction (Row Filters)
- Masking (Column Filters).

#### Phase 1: Establish the Taxonomy & Masking Functions

In [0]:
%sql
-- Create a centralized governance schema
CREATE CATALOG IF NOT EXISTS governance_root;
CREATE SCHEMA IF NOT EXISTS governance_root.policies;

--- 1. MASKING FUNCTION: Credit Cards (Last 4 digits only) ---
CREATE OR REPLACE FUNCTION governance_root.policies.cc_mask(cc STRING)
RETURN IF(is_account_group_member('finance_admin'), cc, concat('****-****-****-', right(cc, 4)));

--- 2. MASKING FUNCTION: Emails (Obfuscate prefix) ---
CREATE OR REPLACE FUNCTION governance_root.policies.email_mask(email STRING)
RETURN IF(is_account_group_member('data_stewards'), email, regexp_replace(email, '^.*@', '****@'));

--- 3. REDACTION FUNCTION: PII/PHI (Full Redaction for unauthorized) ---
CREATE OR REPLACE FUNCTION governance_root.policies.phi_redact(val STRING)
RETURN IF(is_account_group_member('clinical_specialists'), val, 'REDACTED_SENSITIVE_PHI');

--- 4. ROW FILTER: Regional Governance ---
CREATE OR REPLACE FUNCTION governance_root.policies.region_filter(region STRING)
RETURN is_account_group_member(concat('group_', lower(region))) OR is_account_group_member('admin');


#### Phase 2: Create Principals & Assign Permissions

In [0]:
%sql
-- Note: Groups like 'clinical_specialists' and 'finance_admin' must exist in Account Console
-- We grant 'USAGE' first, which is the 'gatekeeper' permission.

-- Public/General Access
GRANT USAGE ON CATALOG main TO `all_users`;

-- Sensitive Data Access (Restricted to specific principals)
GRANT USAGE ON SCHEMA main.hr_data TO `hr_department_gp`;
GRANT SELECT ON SCHEMA main.hr_data TO `hr_department_gp`;

-- Finance Access
GRANT USAGE ON SCHEMA main.finance_data TO `finance_admin`;
GRANT SELECT ON SCHEMA main.finance_data TO `finance_admin`;


#### Phase 3: Robust Tagging & Policy Application Script

In [0]:
from pyspark.sql import SparkSession

def apply_robust_governance(table_fqn, classification, region_col=None):
    """
    Applies a 3-layer security model to a table.
    1. Layer: Tagging (Discovery)
    2. Layer: Row Filtering (Redaction by Region)
    3. Layer: Column Masking (PII Protection)
    """
    
    print(f"--- Applying Governance to {table_fqn} [{classification}] ---")

    # LAYER 1: TAGGING
    spark.sql(f"ALTER TABLE {table_fqn} SET TAGS ('classification' = '{classification}', 'governance_synced' = 'true')")

    # LAYER 2 & 3: SENSITIVITY-BASED POLICIES
    if classification in ["PHI", "PII_HIGH"]:
        # Apply Row Level Redaction if a region column is provided
        if region_col:
            spark.sql(f"ALTER TABLE {table_fqn} SET ROW FILTER governance_root.policies.region_filter ON ({region_col})")
            print(f"Row filter applied on {region_col}")

        # Identify columns to mask/redact based on standard naming conventions
        cols = spark.table(table_fqn).columns
        
        for col in cols:
            if "email" in col.lower():
                spark.sql(f"ALTER TABLE {table_fqn} ALTER COLUMN {col} SET MASK governance_root.policies.email_mask")
            
            elif "credit_card" in col.lower() or "cc_num" in col.lower():
                spark.sql(f"ALTER TABLE {table_fqn} ALTER COLUMN {col} SET MASK governance_root.policies.cc_mask")
                
            elif "ssn" in col.lower() or "health_id" in col.lower():
                spark.sql(f"ALTER TABLE {table_fqn} ALTER COLUMN {col} SET MASK governance_root.policies.phi_redact")
                
    print(f"Governance sync complete for {table_fqn}.\n")

# --- EXECUTION EXAMPLE ---

# Target Table: main.healthcare.patient_records
# Tag: PHI
# Region-based redaction on column 'data_region'
apply_robust_governance("main.healthcare.patient_records", "PHI", region_col="data_region")

# Target Table: main.finance.transactions
# Tag: PII_HIGH
apply_robust_governance("main.finance.transactions", "PII_HIGH")


### Layered Protection Summary

Tagging (Metadata Layer):

- Action: Sets classification=PHI.
- Effect: Data Stewards can run a query to find all PHI tables across the entire metastore for auditing.

Row Filter (Access Layer):

- Action: SET ROW FILTER region_filter.
- Effect: A user in group_uk will never see rows where region = 'US'. The data is redacted at the query level before it reaches the notebook.

Column Masking (Content Layer):
- Action: SET MASK email_mask.
- Effect: Even if the user is allowed to see the row, the email column will return ****@company.com unless they are in the data_stewards group.

### Functions & Policies

In [0]:
%sql
-- Create the filter function first
CREATE OR REPLACE FUNCTION end_to_end_retail.governance_functions.filter_by_city(city STRING)
RETURN city = 'London' OR is_account_group_member('admin');

In [0]:
%sql
CREATE OR REPLACE POLICY `userExampleCityFilter`
ON TABLE end_to_end_retail.db_bronze.tbb_bright_home_orders_enriched
COMMENT 'user example city filter'
ROW FILTER end_to_end_retail.governance_functions.filter_by_city
TO `userExample`
FOR TABLES
MATCH COLUMNS hasTag('pii', 'Tag.Example') AS tagExample
USING COLUMNS (tagExample)